# Run analysis

Plots and tables of one or more runs of the two-site VideoSearcher pipeline. The procedure
that produces a run's directory is in `docs/MULTISITE.md` ("After the run"). In short, each
run is a directory written by

```bash
.venv/bin/python scripts/summarize_run.py results/<run>/logs/edge results/<run>/logs/cloud --out results/<run>
```

with Locust's `request_timestamps.csv` and `workloadProfile.csv` copied next to the CSV files.

List the runs in the next cell and run all the cells (*Run → Run All Cells*). With more than
one run (up to three) every plot compares them. Every figure is also saved as a PNG under
`results/<run>/plots/` (one run) or `results/compare-<runs>/plots/` (several runs).

Times are in seconds (or minutes) from the first job of the run; only what ends within
`DURATION` counts. Some plots follow the ones of the single-site thesis
(ServerlessAutoscalingWithRL, `thesis_experiments/VideoSearcher`), so the results can be
compared with it.

In [ ]:
RUNS = {
    "old agents": "../results/run-1",
}
DURATION = 3600         # seconds of the run
BIN = 60                # seconds per point of the time series
WINDOW = 120            # seconds per point of the per-stage plots (the agents' window)
GLOBAL_THRESHOLD = 200  # seconds: end-to-end threshold used in the single-site thesis
SAVE_PLOTS = True

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Stages in pipeline order, and their response-time thresholds, from the controller's config.
with open(os.path.join("..", "deploy", "k8s", "base", "controller-config.yaml")) as f:
    THRESHOLDS = {s: c.get("res_time_threshold") for s, c in yaml.safe_load(f)["stages"].items()}
STAGES = list(THRESHOLDS)

# Fixed colours: sites and runs keep the same colour in every plot.
SITE_COLORS = {"edge": "#2a78d6", "cloud": "#eb6834"}
RUN_COLORS = dict(zip(RUNS, ["#2a78d6", "#eb6834", "#1baf7a"]))
OBS_COLORS = dict(zip(["workload", "utilization", "pressure", "queue_length_dominant", "n_instances"],
                      ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
INK, MUTED = "#0b0b0b", "#8a8984"
assert len(RUNS) <= 3, "compare at most three runs at a time"
PER_BIN = "minute" if BIN == 60 else f"{BIN:g} s"

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (10, 3.2), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e4e4e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": MUTED, "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "lines.linewidth": 1.5, "legend.frameon": False, "font.size": 9,
})


def slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")


if len(RUNS) == 1:
    PLOT_DIR = os.path.join(next(iter(RUNS.values())), "plots")
else:
    PLOT_DIR = os.path.join("..", "results", "compare-" + "-vs-".join(slug(n) for n in RUNS), "plots")


def save(fig, name):
    if SAVE_PLOTS:
        os.makedirs(PLOT_DIR, exist_ok=True)
        fig.savefig(os.path.join(PLOT_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")


def load(path):
    run = {name: pd.read_csv(os.path.join(path, f"{name}.csv"))
           for name in ("jobs", "stages", "replicas", "agents")}
    requests = os.path.join(path, "request_timestamps.csv")
    if os.path.exists(requests):
        r = pd.read_csv(requests)
        r["t"] = r["Timestamp"] - r["Timestamp"].min()      # from Locust's first request
        run["requests"] = r
    profile = os.path.join(path, "workloadProfile.csv")
    if os.path.exists(profile):
        run["profile"] = pd.read_csv(profile, sep=";")
    s = run["stages"]
    # Response time of a stage: from the previous stage handing the job over to the end.
    s["response_time"] = s["wait"].fillna(0) + s["service_time"]
    # Total service time of every job: the sum of its stages.
    run["jobs"] = run["jobs"].merge(s.groupby("job_id")["service_time"].sum().rename("service_time"),
                                    on="job_id", how="left")
    return run


def completed(run):
    j = run["jobs"]
    return j[(j["outcome"] == "completed") & (j["ended_in_run"] == 1)]


def arrivals_per_bin(run):
    edges = np.arange(0, DURATION + BIN, BIN)
    counts, _ = np.histogram(run["jobs"]["arrival"], bins=edges)
    return edges[:-1], counts


runs = {name: load(path) for name, path in RUNS.items()}
for name, run in runs.items():
    print(f"{name}: {len(run['jobs'])} jobs, sites {sorted(run['stages']['site'].dropna().unique())}, "
          f"Locust files: {[k for k in ('requests', 'profile') if k in run] or 'none'}")
print(f"Plots saved to {os.path.abspath(PLOT_DIR)}" if SAVE_PLOTS else "Plots not saved")

## 1. Load

**What it shows:** jobs per minute that reached the gateway (from the pipeline logs) and,
if Locust's files are there, the rate Locust planned to send.

**How to read it:** the bars should follow the planned curve. Jobs that had not finished at
the end of the run are missing from the bars, so a gap at the very end is expected.

In [ ]:
fig, axes = plt.subplots(len(runs), 1, sharex=True, figsize=(10, 2.6 * len(runs)), squeeze=False)
for ax, (name, run) in zip(axes[:, 0], runs.items()):
    t, counts = arrivals_per_bin(run)
    ax.bar(t / 60, counts, width=BIN / 60 * 0.9, align="edge", color=RUN_COLORS[name],
           label=f"arrived (jobs per {PER_BIN})")
    if "profile" in run:
        p = run["profile"]
        planned = p.groupby(p["Time"] // BIN)["RPS"].sum()
        ax.step(planned.index * BIN / 60, planned.values, where="post", color=INK,
                label=f"planned by Locust (jobs per {PER_BIN})")
    ax.set_title(name, loc="left")
    ax.set_ylabel(f"jobs per {PER_BIN}")
    ax.legend(loc="upper right")
axes[-1, 0].set_xlabel("minutes from the start")
plt.tight_layout()
save(fig, "load")

## 2. End-to-end response time

**What it shows:** for every job, the time from the gateway receiving it to the end of the
last stage (dots), its mean per `WINDOW` (line), and the mean of the sum of the six service
times (the time the job would take with no queue and no hop). The dashed line is the
end-to-end threshold of the single-site thesis.

**How to read it:** the gap between the two lines is time spent waiting (queues, hops between
the sites). If it grows during the peaks of the load, some stage lacks replicas.

In [ ]:
fig, axes = plt.subplots(len(runs), 1, sharex=True, sharey=True, figsize=(10, 3.4 * len(runs)), squeeze=False)
for ax, (name, run) in zip(axes[:, 0], runs.items()):
    j = completed(run)
    ax.scatter(j["arrival"] / 60, j["duration"], s=6, color=RUN_COLORS[name], alpha=0.4, linewidths=0,
               label="response time (each job)")
    w = j.groupby(j["arrival"] // WINDOW)
    x = w.size().index * WINDOW / 60 + WINDOW / 120
    ax.plot(x, w["duration"].mean().values, color=RUN_COLORS[name], marker="o", markersize=3,
            label=f"response time (mean per {WINDOW:g} s)")
    ax.plot(x, w["service_time"].mean().values, color=INK, marker="x", markersize=4,
            label=f"sum of service times (mean per {WINDOW:g} s)")
    ax.axhline(GLOBAL_THRESHOLD, color=INK, linestyle="--", linewidth=1, label=f"threshold ({GLOBAL_THRESHOLD} s)")
    ax.set_title(name, loc="left")
    ax.set_ylabel("seconds")
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
axes[-1, 0].set_xlabel("arrival (minutes from the start)")
plt.tight_layout()
save(fig, "response_time")

## 3. Distribution of the response time

**What it shows:** the share of jobs that finished within a given time (CDF), with its
median and 95th percentile, and a table per run.

**How to read it:** the steeper the curve, the more regular the pipeline; a long tail on the
right means some jobs waited much longer than the others (queues at the peaks).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for name, run in runs.items():
    d = np.sort(completed(run)["duration"])
    if not len(d):
        continue
    ax.plot(d, np.arange(1, len(d) + 1) / len(d) * 100, color=RUN_COLORS[name], label=name)
    for q, style in ((50, ":"), (95, "--")):
        ax.axvline(np.percentile(d, q), color=RUN_COLORS[name], linestyle=style, linewidth=1)
ax.set_xlabel("job response time (s)")
ax.set_ylabel("% of jobs")
ax.set_title("Response time CDF (dotted: p50, dashed: p95)", loc="left")
if len(runs) > 1:
    ax.legend()
plt.tight_layout()
save(fig, "response_time_cdf")

rows = {}
for name, run in runs.items():
    j = run["jobs"]
    done = completed(run)["duration"]
    rows[name] = {
        "completed in the run": len(done),
        "completed after the end": int(((j["outcome"] == "completed") & (j["ended_in_run"] == 0)).sum()),
        "failed": int((j["outcome"] == "failed").sum()),
        **{f"p{q} (s)": done.quantile(q / 100) for q in (50, 90, 95, 99)},
        "mean (s)": done.mean(),
        "max (s)": done.max(),
        f"over {GLOBAL_THRESHOLD} s (%)": (done > GLOBAL_THRESHOLD).mean() * 100,
    }
pd.DataFrame(rows).T.round(2)

## 4. Response and service time of every stage

**What it shows:** for every stage, the mean response time (from the previous stage handing
the job over to the end of this one: queue, hop, seed copy and service) and the mean service
time per `WINDOW`, with the stage's `res_time_threshold` (dashed). It is the per-function
plot of the single-site thesis.

**How to read it:** where the response time leaves the service time, jobs waited for that
stage. Above the dashed line the stage violates its threshold, which is what the agents try
to avoid.

In [ ]:
fig, axes = plt.subplots(2, 3, sharex=True, figsize=(13, 6.4))
for ax, stage in zip(axes.flat, STAGES):
    for name, run in runs.items():
        s = run["stages"][run["stages"]["stage"] == stage]
        w = s.groupby(s["end"] // WINDOW)
        x = w.size().index * WINDOW / 60 + WINDOW / 120
        ax.plot(x, w["response_time"].mean().values, color=RUN_COLORS[name], marker="o", markersize=3,
                label=f"response ({name})" if len(runs) > 1 else "response time")
        ax.plot(x, w["service_time"].mean().values, color=RUN_COLORS[name], linestyle="--", linewidth=1,
                label=f"service ({name})" if len(runs) > 1 else "service time")
    if THRESHOLDS.get(stage):
        ax.axhline(THRESHOLDS[stage], color=INK, linestyle="--", linewidth=1, label="threshold")
    ax.set_title(stage, loc="left")
    ax.set_ylim(bottom=0)
for ax in axes[:, 0]:
    ax.set_ylabel("seconds")
for ax in axes[-1]:
    ax.set_xlabel("minutes from the start")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
save(fig, "stages_response_service")

## 5. Service time of every stage (profiling)

**What it shows:** the time each stage spent working on a job (end - start in the worker),
per site: a table and box plots.

**How to read it:** the means are the *demands* of the training environment of the agents.
The single-site thesis set every `res_time_threshold` to 5 × the median service time
(`parse_complete.ipynb`): the last column applies the same rule to this run, next to the
threshold in use.

In [ ]:
table = pd.concat({name: run["stages"] for name, run in runs.items()}, names=["run"]).reset_index(level=0)
summary = (table.groupby(["stage", "site", "run"])["service_time"]
           .describe(percentiles=[0.05, 0.5, 0.95])
           .rename(columns={"count": "n"}))
summary = summary.reindex(STAGES, level="stage")
summary["threshold in use"] = [THRESHOLDS.get(stage) for stage, _, _ in summary.index]
summary["5 x median"] = summary["50%"] * 5
summary.round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for ax, stage in zip(axes.flat, STAGES):
    data, labels, colors = [], [], []
    for name, run in runs.items():
        s = run["stages"][run["stages"]["stage"] == stage]
        for site in ("edge", "cloud"):
            values = s.loc[s["site"] == site, "service_time"]
            if len(values):
                data.append(values)
                labels.append(site if len(runs) == 1 else f"{site}\n{name}")
                colors.append(SITE_COLORS[site])
    if data:
        box = ax.boxplot(data, patch_artist=True, widths=0.6,
                         flierprops={"markersize": 3, "markeredgecolor": MUTED},
                         medianprops={"color": INK})
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_edgecolor(color)
        ax.set_xticks(range(1, len(labels) + 1), labels)
    ax.set_title(stage, loc="left")
for ax in axes[:, 0]:
    ax.set_ylabel("service time (s)")
plt.tight_layout()
save(fig, "service_time")

## 6. Service time against wait

**What it shows:** every job at every stage as a point (service time, wait before the stage
started it).

**How to read it:** points near the horizontal axis mean the stage always had a free replica.
Points high up mean jobs queued for that stage: it is the one to scale.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for ax, stage in zip(axes.flat, STAGES):
    for name, run in runs.items():
        s = run["stages"][run["stages"]["stage"] == stage]
        ax.scatter(s["service_time"], s["wait"], s=6, alpha=0.4, linewidths=0, color=RUN_COLORS[name], label=name)
    ax.set_title(stage, loc="left")
for ax in axes[-1]:
    ax.set_xlabel("service time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("wait (s)")
if len(runs) > 1:
    for handle in axes[0, 0].legend(markerscale=2).legend_handles:
        handle.set_alpha(1)
plt.tight_layout()
save(fig, "service_vs_wait")

## 7. Waits per stage over time

**What it shows:** the mean wait of the jobs entering each stage, per `BIN`: queueing, plus
the hop from the previous stage (and the seed copy when the job comes from the other site).

**How to read it:** a wait that rises with the load and falls back means the scaling reacted;
one that keeps rising means the stage did not get enough replicas in time.

In [ ]:
fig, axes = plt.subplots(2, 3, sharex=True, figsize=(12, 5.2))
for ax, stage in zip(axes.flat, STAGES):
    for name, run in runs.items():
        s = run["stages"][run["stages"]["stage"] == stage]
        w = s.groupby(s["start"] // BIN)["wait"].mean()
        ax.plot(w.index * BIN / 60, w.values, color=RUN_COLORS[name], label=name)
    ax.set_title(stage, loc="left")
for ax in axes[:, 0]:
    ax.set_ylabel("mean wait (s)")
for ax in axes[-1]:
    ax.set_xlabel("minutes from the start")
if len(runs) > 1:
    axes[0, 0].legend()
plt.tight_layout()
save(fig, "waits")

## 8. Replicas

**What it shows:** for every stage and site, the replicas alive (solid: running, including
those shutting down but still working) and desired (dashed: what the controller asked
for), with the load on top. One figure per run.

**How to read it:** the replicas should rise with the load and fall in the 20 minutes without
load. A dashed line above the solid one for long means pods could not start (e.g. not
enough CPU on the node).

In [ ]:
for name, run in runs.items():
    fig, axes = plt.subplots(len(STAGES) + 1, 1, sharex=True, figsize=(10, 1.5 * (len(STAGES) + 1)))
    t, counts = arrivals_per_bin(run)
    axes[0].bar(t / 60, counts, width=BIN / 60 * 0.9, align="edge", color=MUTED)
    axes[0].set_ylabel(f"jobs per {PER_BIN}")
    axes[0].set_title(f"{name}: load and replicas", loc="left")
    r = run["replicas"]
    for ax, stage in zip(axes[1:], STAGES):
        for site, color in SITE_COLORS.items():
            s = r[(r["stage"] == stage) & (r["site"] == site)]
            if s["alive"].any() or s["desired"].any():
                ax.step(s["t"] / 60, s["alive"], where="post", color=color)
                ax.step(s["t"] / 60, s["desired"], where="post", color=color, linestyle="--", linewidth=1)
        ax.set_ylabel(stage, rotation=0, ha="right", va="center")
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    handles = [plt.Line2D([], [], color=c, label=s) for s, c in SITE_COLORS.items()]
    handles += [plt.Line2D([], [], color=MUTED, label="alive"),
                plt.Line2D([], [], color=MUTED, linestyle="--", linewidth=1, label="desired")]
    axes[1].legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    axes[-1].set_xlabel("minutes from the start")
    axes[-1].set_xlim(0, DURATION / 60)
    plt.tight_layout()
    save(fig, f"replicas_{slug(name)}")

## 9. Agents

**What it shows:** for every stage scaled by an agent, the normalised state it received every
`WINDOW` (all between 0 and 1), its answers (points) and the replicas applied after each one
(kept between the `min` and `max` of the stage). One figure per run and stage.

**How to read it:** the agent should add replicas when pressure and queue length rise, and
remove them when the workload drops.

In [ ]:
for name, run in runs.items():
    a = run["agents"]
    decided = a[a["action"].notna()]
    if decided.empty:
        print(f"{name}: no agent decisions")
        continue
    for (stage, site), d in decided.groupby(["stage", "site"]):
        fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(10, 3.6), height_ratios=[3, 2])
        for key, color in OBS_COLORS.items():
            top.plot(d["t"] / 60, d[f"obs_{key}"], color=color, marker="o", markersize=3, label=key)
        top.set_ylim(-0.05, 1.05)
        top.set_ylabel("state (0-1)")
        top.set_title(f"{name}: agent of {stage} on {site}", loc="left")
        top.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
        bottom.step(d["t"] / 60, d["applied"], where="post", color=SITE_COLORS[site], label="replicas applied")
        bottom.plot(d["t"] / 60, d["action"], "o", color=INK, markersize=4, label="agent's answer")
        bottom.set_ylabel("replicas")
        bottom.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        bottom.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
        bottom.set_xlabel("minutes from the start")
        plt.tight_layout()
        save(fig, f"agent_{slug(name)}_{stage}_{site}")

## 10. Resources and threshold violations

**What it shows:** the replica-hours used by every stage on every site within the run (from
the replicas alive, sampled every 5 s), and the share of jobs whose response time at a
stage exceeded its `res_time_threshold` by more than 10 % (how the original controller
counts a violation).

**How to read it:** together they give the trade-off of a scaling policy: fewer
replica-hours and fewer violations are both better.

In [ ]:
def replica_hours(run):
    r = run["replicas"].sort_values("t")
    r = r[(r["t"] >= 0) & (r["t"] <= DURATION)].copy()
    r["dt"] = r.groupby(["site", "stage"])["t"].diff().shift(-1).fillna(0)
    return (r["alive"] * r["dt"]).groupby([r["stage"], r["site"]]).sum() / 3600


def violations(run):
    s = run["stages"]
    limit = s["stage"].map(THRESHOLDS) * 1.1
    return (s["response_time"] > limit).groupby(s["stage"]).mean() * 100


hours = pd.DataFrame({name: replica_hours(run) for name, run in runs.items()}).reindex(STAGES, level=0)
viol = pd.DataFrame({name: violations(run) for name, run in runs.items()}).reindex(STAGES)
print("Replica-hours")
display(hours.round(2))
print("Jobs over the threshold (%)")
display(viol.round(1))

## 11. Comparison between runs

**What it shows:** the key numbers of every run side by side, and the resources used per
stage.

In [ ]:
compare = {}
for name, run in runs.items():
    j = run["jobs"]
    done = completed(run)
    compare[name] = {
        "jobs completed in the run": len(done),
        "failed": int((j["outcome"] == "failed").sum()),
        "response time p50 (s)": done["duration"].median(),
        "response time p95 (s)": done["duration"].quantile(0.95),
        f"over {GLOBAL_THRESHOLD} s (%)": (done["duration"] > GLOBAL_THRESHOLD).mean() * 100,
        "replica-hours": replica_hours(run).sum(),
        **{f"violations {s} (%)": violations(run).get(s, np.nan) for s in ("ffmpeg2", "deepspeech")},
    }
pd.DataFrame(compare).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
per_stage = hours.groupby(level="stage").sum().reindex(STAGES)
width = 0.8 / len(runs)
for i, name in enumerate(runs):
    ax.bar(np.arange(len(STAGES)) + i * width, per_stage[name], width=width * 0.92,
           color=RUN_COLORS[name], label=name)
ax.set_xticks(np.arange(len(STAGES)) + width * (len(runs) - 1) / 2, STAGES)
ax.set_ylabel("replica-hours")
ax.set_title("Resources used per stage (both sites)", loc="left")
if len(runs) > 1:
    ax.legend()
plt.tight_layout()
save(fig, "replica_hours")